Decision Tree Classifier

Build a model that predicts the Drug to be prescribed to the patients on basis of their medical results

Step1: Data Gathering

In [ ]:
import pandas as pd
path = r"https://raw.githubusercontent.com/sindhura-nk/Datasets/refs/heads/main/drug200.csv"
df = pd.read_csv(path)

In [ ]:
df.head()

Step2: Perform the basic data quality checks

In [ ]:
df.shape

In [ ]:
df.info()

In [1]:
df['Drug'].unique()

NameError: name 'df' is not defined

In [ ]:
## Check for duplicated information
df.duplicated().sum()

In [ ]:
df=df.drop_duplicates()

In [ ]:
#Check for missing data
df.isna().sum()

Separate X and Y features
X: all features except Drug
Y: Drug

In [ ]:
X = df.drop(columns=['Drug'])
Y = df[['Drug']]

In [ ]:
X.head()    

In [ ]:
Y.head()

Feature Engineering: Data cleaning, Data preprocessing(Feature Scaling)

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.compose import ColumnTransformer

cat = list(X.select_dtypes(include='str').columns)
con = list(X.select_dtypes(include='number').columns)


num_pipe = make_pipeline(
    SimpleImputer(strategy='mean'),
    StandardScaler()
)

cat_pipe = make_pipeline(
    SimpleImputer(strategy='most_frequent'),
    OneHotEncoder(handle_unknown='ignore',sparse_output=False)
)

pre = ColumnTransformer([
    ('cat',cat_pipe,cat),
    ('con',num_pipe,con)
]).set_output(transform='pandas')

X_pre = pre.fit_transform(X)
X_pre.head()

Split the data into training and testing

In [ ]:
from sklearn.model_selection import train_test_split
xtrain,xtest,ytrain,ytest = train_test_split(X_pre,Y,train_size=0.66,random_state=21)

In [ ]:
print(f"xtrain:{xtrain.shape}")
print(f"xtest:{xtest.shape}")
print(f"ytrain:{ytrain.shape}")
print(f"ytest:{ytest.shape}")

Model building- decision tree model

In [ ]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(
    criterion='gini',
    max_depth=5,
    min_samples_leaf=3,
    min_samples_split=5
)

model.fit(xtrain,ytrain)

In [ ]:

model.score(xtrain,ytrain)

In [ ]:
model.score(xtest,ytest)

Hyperparameter Tuning

criterion='gini',
max_depth=5,
min_samples_leaf=3,
min_samples_split=5

In [ ]:
params ={
    'max_depth':[3,4,5,6,7],
    'min_samples_leaf':[1,2,3,4,5,9,11],
    'min_samples_split':[3,5,7,9,10],
    'criterion':['gini','entropy']
}

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
base_model = DecisionTreeClassifier()
rscv = RandomizedSearchCV(estimator=base_model,param_distributions=params,cv=3,scoring='f1_macro')
rscv.fit(xtrain,ytrain)

Model evaluation

In [ ]:

rscv.best_params_

In [ ]:
rscv.best_score_

In [ ]:
best_dtc = rscv.best_estimator_
best_dtc

In [ ]:

best_dtc.score(xtrain,ytrain)

In [ ]:

best_dtc.score(xtest,ytest)


Confusion Matrix and Classification Report

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
ConfusionMatrixDisplay.from_estimator(best_dtc,xtest,ytest)

In [ ]:
from sklearn.metrics import classification_report
ypreds = model.predict(xtest)
print(classification_report(ytest,ypreds))

In [ ]:
from sklearn.metrics import f1_score
f1 = f1_score(ytest,ypreds,average='macro')
f1

Plot the decision tree

In [ ]:
xtrain.columns

In [ ]:
from sklearn.tree import plot_tree
fea_names = xtrain.columns

import matplotlib.pyplot as plt
plt.figure(figsize=(25,25))
plot_tree(best_dtc,feature_names=fea_names,class_names=best_dtc.classes_,filled=True)
plt.show()


Important features considered by the best_dtc model

In [ ]:

best_dtc.feature_importances_

In [ ]:
best_dtc.classes_

In [ ]:
xtrain.columns

In [ ]:
pd.Series(best_dtc.feature_importances_,index=xtrain.columns)

In [ ]:
A = pd.Series(best_dtc.feature_importances_,index=xtrain.columns)
A.sort_values()

Save the model and we can save the preprocessor pipeline

In [ ]:
import joblib
joblib.dump(pre,'pre.joblib')
joblib.dump(best_dtc,'model.joblib')

In [ ]:
pre2 = joblib.load('pre.joblib')
model2 = joblib.load('model.joblib')

Out of sample predictions

In [ ]:
path2 = r"https://raw.githubusercontent.com/sindhura-nk/Datasets/refs/heads/main/DrugTest.csv"
xnew = pd.read_csv(path2)
xnew.head()

In [ ]:
# prepare the data
xnew_pre = pre2.transform(xnew)
xnew_pre.head()

In [ ]:
preds = model2.predict(xnew_pre)
preds

In [ ]:
xnew['Drug_Predictions'] = preds
xnew

In [ ]:
## Save the file
xnew.to_csv("Drug Predictions.csv",index=False)